<a href="https://colab.research.google.com/github/KarimBekhtiGalvao/Diffusion-Based-Generation/blob/main/Generation_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Architecture from github

In [4]:
import functools

import fire
import numpy as np
import tensorflow.compat.v1 as tf

from diffusion_tf import utils
from diffusion_tf.diffusion_utils import get_beta_schedule, GaussianDiffusion
from diffusion_tf.models import unet
from diffusion_tf.tpu_utils import tpu_utils, datasets


class Model(tpu_utils.Model):
  def __init__(self,
               *,
               model_name,
               betas: np.ndarray,
               loss_type: str,
               num_classes: int,
               dropout: float,
               randflip,
               block_size: int):
    self.model_name = model_name
    self.diffusion = GaussianDiffusion(betas=betas, loss_type=loss_type)
    self.num_classes = num_classes
    self.dropout = dropout
    self.randflip = randflip
    self.block_size = block_size

  def _denoise(self, x, t, y, dropout):
    B, H, W, C = x.shape.as_list()
    assert x.dtype == tf.float32
    assert t.shape == [B] and t.dtype in [tf.int32, tf.int64]
    assert y.shape == [B] and y.dtype in [tf.int32, tf.int64]
    orig_out_ch = out_ch = C

    if self.block_size != 1:  # this can be used to reduce memory consumption
      x = tf.nn.space_to_depth(x, self.block_size)
      out_ch *= self.block_size ** 2

    y = None
    if self.model_name == 'unet2d16b2c112244':  # 114M for block_size=1
      out = unet.model(
        x, t=t, y=y, name='model', ch=128, ch_mult=(1, 1, 2, 2, 4, 4), num_res_blocks=2, attn_resolutions=(16,),
        out_ch=out_ch, num_classes=self.num_classes, dropout=dropout
      )
    else:
      raise NotImplementedError(self.model_name)

    if self.block_size != 1:
      out = tf.nn.depth_to_space(out, self.block_size)
    assert out.shape == [B, H, W, orig_out_ch]
    return out

  def train_fn(self, x, y):
    B, H, W, C = x.shape
    if self.randflip:
      x = tf.image.random_flip_left_right(x)
      assert x.shape == [B, H, W, C]
    t = tf.random_uniform([B], 0, self.diffusion.num_timesteps, dtype=tf.int32)
    losses = self.diffusion.p_losses(
      denoise_fn=functools.partial(self._denoise, y=y, dropout=self.dropout), x_start=x, t=t)
    assert losses.shape == t.shape == [B]
    return {'loss': tf.reduce_mean(losses)}

  def samples_fn(self, dummy_noise, y):
    return {
      'samples': self.diffusion.p_sample_loop(
        denoise_fn=functools.partial(self._denoise, y=y, dropout=0),
        shape=dummy_noise.shape.as_list(),
        noise_fn=tf.random_normal
      )
    }

  def samples_fn_denoising_trajectory(self, dummy_noise, y, repeat_noise_steps=0):
    times, imgs = self.diffusion.p_sample_loop_trajectory(
      denoise_fn=functools.partial(self._denoise, y=y, dropout=0),
      shape=dummy_noise.shape.as_list(),
      noise_fn=tf.random_normal,
      repeat_noise_steps=repeat_noise_steps
    )
    return {
      'samples': imgs[-1],
      'denoising_trajectory_times': times,
      'denoising_trajectory_images': imgs
    }

  def interpolate_fn(self, dummy_noise, y):
    x1, x2, lam, x_interp, t = self.diffusion.interpolate(
      denoise_fn=functools.partial(self._denoise, y=y, dropout=0),
      shape=dummy_noise.shape.as_list(),
      noise_fn=tf.random_normal,
    )
    return {
      'x1': x1,    # placeholder
      'x2': x2,    # placeholder
      'lam': lam,  # placeholder
      't': t,      # placeholder
      'x_interp': x_interp
    }


ModuleNotFoundError: No module named 'diffusion_tf'

In [5]:
!pip install diffusion_tf

ERROR: Could not find a version that satisfies the requirement diffusion_tf (from versions: none)
ERROR: No matching distribution found for diffusion_tf
